In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document


In [4]:
import os
from dotenv import load_dotenv

#  Load API keys
load_dotenv(".env")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# 1. Load the text file
with open("sample.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# 2. Split the text into chunks
splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_text(raw_text)
documents = [Document(page_content=chunk) for chunk in chunks]

# 3. Create vector store
embedding = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(documents, embedding=embedding)

# 4. Setup LLM
# NOTE: the original notebook relied on the default model (gpt-3.5-turbo era),
# which has since been retired — pin an explicit, currently-supported model.
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

In [6]:
## 1) Retrieve-only contextual compression
# Get retrieved docs --> it does not generate an answer, it only compresses/filters them
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

# Compressor using LLM
compressor = LLMChainExtractor.from_llm(llm)

# Create compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectorstore.as_retriever()
)

# Run retrieval (.invoke replaces the deprecated .get_relevant_documents())
print("Contextual Compression Results:")
results = compression_retriever.invoke("Who created LangChain?")
for doc in results:
    print("-", doc.page_content)

Contextual Compression Results:
- LangChain was created by Harrison Chase.


In [7]:
## 2) Conversational RAG with memory — without agent mode
# Integration with a conversational RAG "chain" — without agent mode
# (ConversationalRetrievalChain is deprecated; this is the LangChain-recommended
#  manual replacement: explicit retrieval + an explicit chat-history list)

chat_history: list[tuple[str, str]] = []  # [(question, answer), ...]

def rag_chain_fn(question: str) -> str:
    docs = compression_retriever.invoke(question)
    context = "\n".join(doc.page_content for doc in docs)

    history_text = "\n".join(f"Q: {q}\nA: {a}" for q, a in chat_history)
    prompt = (
        "Answer the question using the conversation history and context below.\n\n"
        f"Conversation history:\n{history_text}\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}"
    )

    answer = llm.invoke(prompt).content
    chat_history.append((question, answer))
    return answer

print("ConversationalRetrievalChain (no agent):")
print(rag_chain_fn("What is LangChain?"))
print(rag_chain_fn("Who created it?"))

ConversationalRetrievalChain (no agent):
LangChain is a framework for building applications with large language models (LLMs). It was created by Harrison Chase and supports features such as retrieval-augmented generation (RAG), agents, memory, tools, and more. LangChain is commonly used in chatbots, document question answering, and AI workflows.
LangChain was created by Harrison Chase.


In [8]:
## 3) Integration into an agent (with a tool)
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

@tool
def RAG_Tool(question: str) -> str:
    """Answer LangChain-related questions with context."""
    docs = compression_retriever.invoke(question)
    context = "\n".join(doc.page_content for doc in docs)
    answer = llm.invoke(
        f"Answer the question using only the context below.\n\nContext:\n{context}\n\nQuestion: {question}"
    )
    return answer.content

checkpointer = InMemorySaver()
thread_config = {"configurable": {"thread_id": "compression-demo-1"}}

agent = create_agent(
    model=llm,
    tools=[RAG_Tool],
    checkpointer=checkpointer,
)

print("\n Agent Conversation:")
res1 = agent.invoke({"messages": [{"role": "user", "content": "What is LangChain?"}]}, thread_config)
print(res1["messages"][-1].content)

res2 = agent.invoke({"messages": [{"role": "user", "content": "Who created it?"}]}, thread_config)
print(res2["messages"][-1].content)


 Agent Conversation:
LangChain is a framework designed for building applications that utilize large language models (LLMs). It provides tools and components to help developers create applications that leverage the capabilities of these models effectively. If you want, I can provide more detailed information about its features and use cases.
LangChain was created by Harrison Chase. If you want to know more about the creator or the history of LangChain, feel free to ask!


In [9]:
## 4) MultiQueryRetriever
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

# MultiQueryRetriever (still available via langchain-classic)
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(),
    llm=llm
)

# Retrieve + generate directly (replaces the deprecated RetrievalQA chain)
query = "Tell me about LangChain creator and features."
source_docs = multi_query_retriever.invoke(query)
context = "\n".join(doc.page_content for doc in source_docs)

answer = llm.invoke(
    f"Answer the question using only the context below.\n\nContext:\n{context}\n\nQuestion: {query}"
).content

print("\n RAG Pipeline (MultiQueryRetriever):")
print("Answer:", answer)

print("\nSources:")
for doc in source_docs:
    print("-", doc.page_content[:200])  # print first 200 chars of each doc


 RAG Pipeline (MultiQueryRetriever):
Answer: LangChain was created by Harrison Chase. Its features include support for RAG (Retrieval-Augmented Generation), agents, memory, tools, and more.

Sources:
- LangChain is a framework for building applications with LLMs.LangChain was created by Harrison Chase.LangChain supports RAG, agents, memory, tools, and more.It’s commonly used in chatbots, document Q&
